# Chapter 13 — Introducing Automatic Optimization

## Learning Objectives

This notebook covers tensors, computation graphs, autograd, automatic backpropagation, and a minimal neural network framework abstraction.

## Theoretical Explanation

Deep learning frameworks automate gradient computation. Instead of manually deriving every gradient, tensors record how they were created. During backward propagation, gradients move through the computation graph using each operation's local backward rule.

Building a tiny framework helps explain how PyTorch-like systems work: tensors store data, gradients, creator tensors, and the operation that created them. This abstraction allows the user to focus mostly on forward propagation.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Minimal Autograd for Addition and Multiplication

In [2]:
class Tensor:
    def __init__(self, data, creators=None, creation_op=None):
        self.data = np.array(data, dtype=float)
        self.creators = creators
        self.creation_op = creation_op
        self.grad = np.zeros_like(self.data)
    def __add__(self, other):
        return Tensor(self.data + other.data, [self, other], 'add')
    def __mul__(self, other):
        return Tensor(self.data * other.data, [self, other], 'mul')
    def backward(self, grad=None):
        if grad is None:
            grad = np.ones_like(self.data)
        self.grad += grad
        if self.creation_op == 'add':
            self.creators[0].backward(grad)
            self.creators[1].backward(grad)
        if self.creation_op == 'mul':
            a, b = self.creators
            a.backward(grad * b.data)
            b.backward(grad * a.data)

x = Tensor([2.0])
y = Tensor([3.0])
z = x * y + y
z.backward()
print('z data:', z.data)
print('dz/dx:', x.grad)
print('dz/dy:', y.grad)

z data: [9.]
dz/dx: [3.]
dz/dy: [3.]


### Output Interpretation

The mini tensor object tracks operations and computes gradients backward. This is the core idea behind automatic differentiation.

## Extended Study Notes

The main learning style in this notebook is intentionally close to the book's philosophy: build the idea from small numerical operations, inspect the output, and then connect the output back to the deep learning concept. Instead of treating a neural network as a black box, each notebook exposes the role of inputs, weights, predictions, error, deltas, gradients, and updates.

The examples are original/adapted demonstrations written for academic learning. They preserve the conceptual workflow of the reference book while avoiding direct copying of long code listings. This is important for academic integrity and also makes the notebooks easier to understand as independent study material.

## Chapter Summary

This chapter was reproduced as a compact but complete study notebook. It combines conceptual explanation, NumPy-based implementation, output interpretation, and practical deep learning context.

## Key Takeaways

- Neural networks are built from repeated numerical operations on vectors, matrices, and tensors.
- Learning means changing weights so predictions produce smaller error.
- Shape consistency and interpretation of intermediate values are essential for debugging.
- Understanding the from-scratch implementation makes high-level frameworks easier to use responsibly.